# Polish prompt evaluation

Loads the runs in `eval/runs/` written by `run.py` and charts them per variant and per tag. Run from `eval/`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from metrics import summarize, by_tag, score_rows

runs = sorted(Path('runs').glob('*.parquet'))
runs[-5:]

In [ ]:
# Pick the run(s) to compare; label each by variant + schema + guardrails
selected = [runs[-1]]
df = pd.concat([pd.read_parquet(p) for p in selected], ignore_index=True)
df['tags'] = df['tags'].apply(list)
for col in ('problems', 'final_problems'):
    df[col] = df[col].apply(list)
df['label'] = df['variant'] + '/' + df['schema'] + '/' + df['guardrails']
summarize(df, by='label')

In [ ]:
s = summarize(df, by='label').T
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
s[['placeholder_integrity_first', 'unchanged_when_correct']].plot.bar(ax=axes[0], ylim=(0.5, 1.0), title='Integrity and unchanged-when-correct')
s[['wer_fix_rows']].plot.bar(ax=axes[1], title='WER vs expected (fix rows, lower is better)', legend=False)
s[['latency_p50_s', 'latency_p90_le60w_s']].plot.bar(ax=axes[2], title='Latency (s)')
for ax in axes: ax.tick_params(axis='x', rotation=30)
plt.tight_layout()

In [ ]:
tags = score_rows(df).explode('tags').pivot_table(index='tags', columns='label', values='wer', aggfunc='mean')
tags.plot.barh(figsize=(10, 6), title='WER vs expected by tag')
tags

In [ ]:
# Rows where the model broke a placeholder or the pipeline fell back
scored = score_rows(df)
scored[(~scored.ph_ok_first) | scored.fallback][['label', 'id', 'problems', 'final_problems', 'error', 'raw']]

In [ ]:
# Correct rows the model changed anyway
scored[scored.is_correct_row & ~scored.unchanged][['label', 'id', 'input', 'output']]